# Phase 3 — Price Cannibalization Analysis

This notebook documents the relationship between renewable penetration and a gas-implied electricity price proxy. Because the dependent variable is mechanically derived from Henry Hub gas prices, it is not a valid standalone test of ERCOT price cannibalization; actual ERCOT hub and LMP prices would be required for that causal/statistical claim.

In [1]:

import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import statsmodels.api as sm
from scipy import stats

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CHROME_PATH = PROJECT_ROOT / '.kaleido_chrome/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing'
CHART_DIR = PROJECT_ROOT / 'outputs/charts'
CHART_DIR.mkdir(parents=True, exist_ok=True)

def apply_chart_standard(fig, title, xaxis_title, yaxis_title, source='Source: U.S. EIA; ERCOT; author calculations'):
    fig.update_layout(
        title=dict(text=title, font=dict(size=18, family='Arial', color='black')),
        font=dict(family='Arial', size=12),
        xaxis_title=xaxis_title,
        yaxis_title=yaxis_title,
        plot_bgcolor='white',
        paper_bgcolor='white',
        hovermode='closest',
        margin=dict(l=70, r=70, t=90, b=80),
    )
    fig.update_xaxes(showgrid=True, gridcolor='#F5F5F5', title_font=dict(size=14))
    fig.update_yaxes(showgrid=True, gridcolor='#F5F5F5', title_font=dict(size=14))
    fig.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.14, text=source,
                       showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
    return fig

def save_chart(fig, filename):
    html_path = CHART_DIR / f'{filename}.html'
    png_path = CHART_DIR / f'{filename}.png'
    fig.write_html(html_path)
    print(f'Saved: {html_path.relative_to(PROJECT_ROOT)}')
    subprocess.run([
        str(CHROME_PATH), '--headless=new', '--disable-gpu', '--no-sandbox', '--disable-dev-shm-usage',
        f'--screenshot={png_path}', '--window-size=1200,700', f'file://{html_path}'
    ], check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    print(f'Saved: {png_path.relative_to(PROJECT_ROOT)}')

price_df = pd.read_csv(PROJECT_ROOT / 'data/processed/price_analysis.csv', parse_dates=['month_date'])
print(f'Loaded price analysis dataset: {price_df.shape}')
print(price_df.head(10).to_string(index=False))
print('Chart export method: Plotly HTML + approved headless Chrome PNG screenshot')


Loaded price analysis dataset: (120, 20)
 year  month month_date  total_generation_mwh  renewable_mwh  wind_mwh  solar_mwh    gas_mwh   coal_mwh  renewable_pct  solar_pct  wind_pct  renewable_pct_prior_year  renewable_yoy_change   season  peak_season_flag  henry_hub_price  henry_hub_max  henry_hub_min  implied_wholesale_price_proxy
 2015      1 2015-01-01            38164000.0      3063000.0 3031000.0    32000.0 19252000.0 11365000.0          8.026      0.084     7.942                       NaN                   NaN   winter                 1         2.994500           3.32           2.88                      52.403750
 2015      2 2015-02-01            33449000.0      3301000.0 3268000.0    33000.0 16980000.0  9146000.0          9.869      0.099     9.770                       NaN                   NaN   winter                 1         2.874737           3.22           2.62                      50.307895
 2015      3 2015-03-01            33042000.0      2586000.0 2544000.0    42000.

In [2]:

print('PRICE CANNIBALIZATION ANALYSIS')
print('=' * 50)

corr_overall = price_df['renewable_pct'].corr(price_df['implied_wholesale_price_proxy'])
print(f'Overall correlation: renewable share vs electricity price proxy: {corr_overall:.3f}')

pearson_r, pearson_p = stats.pearsonr(price_df['renewable_pct'], price_df['implied_wholesale_price_proxy'])
spearman_r, spearman_p = stats.spearmanr(price_df['renewable_pct'], price_df['implied_wholesale_price_proxy'])
print(f'Pearson r: {pearson_r:.3f}, p={pearson_p:.4f}')
print(f'Spearman rho: {spearman_r:.3f}, p={spearman_p:.4f}')

print('\nCorrelation by year:')
for year in range(2018, 2025):
    yr_data = price_df[price_df['year'] == year]
    if len(yr_data) > 6:
        corr = yr_data['renewable_pct'].corr(yr_data['implied_wholesale_price_proxy'])
        print(f'  {year}: {corr:.3f}')

print('\nCorrelation by season:')
for season in ['summer', 'winter', 'shoulder']:
    s_data = price_df[price_df['season'] == season]
    corr = s_data['renewable_pct'].corr(s_data['implied_wholesale_price_proxy'])
    print(f'  {season}: {corr:.3f} (n={len(s_data)})')


PRICE CANNIBALIZATION ANALYSIS
Overall correlation: renewable share vs electricity price proxy: 0.122
Pearson r: 0.122, p=0.1859
Spearman rho: -0.051, p=0.5793

Correlation by year:
  2018: -0.016
  2019: 0.351
  2020: 0.119
  2021: -0.339
  2022: -0.357
  2023: -0.034
  2024: -0.557

Correlation by season:
  summer: 0.167 (n=30)
  winter: 0.040 (n=30)
  shoulder: 0.170 (n=60)


In [3]:

reg_df = price_df.dropna(subset=['renewable_pct', 'henry_hub_price', 'implied_wholesale_price_proxy']).copy()
reg_df['summer'] = (reg_df['season'] == 'summer').astype(int)
reg_df['winter'] = (reg_df['season'] == 'winter').astype(int)
reg_df['time_trend'] = (reg_df['year'] - 2015) * 12 + reg_df['month']

X1 = sm.add_constant(reg_df['renewable_pct'])
y = reg_df['implied_wholesale_price_proxy']
model1 = sm.OLS(y, X1).fit(cov_type='HC1')

X2 = sm.add_constant(reg_df[['renewable_pct', 'henry_hub_price', 'summer', 'winter', 'time_trend']])
model2 = sm.OLS(y, X2).fit(cov_type='HC1')

print('MODEL 1: Simple OLS — Renewable Share -> Electricity Price Proxy')
print(model1.summary())
print('\n')
print('MODEL 2: Full OLS — Renewable Share + Gas Price + Season + Trend')
print(model2.summary())

coef = model2.params['renewable_pct']
pval = model2.pvalues['renewable_pct']
print('\nKEY FINDING:')
print(f'  Coefficient on renewable_pct: {coef:.3f}')
print('  Interpretation: A 1 percentage point increase in renewable share')
print(f'  is associated with a ${coef:.2f}/MWh change in gas-implied wholesale electricity price')
print(f'  P-value: {pval:.4f} ({"statistically significant" if pval < 0.05 else "not significant at 5% level"})')
print(f'  R-squared: {model2.rsquared:.3f}')
print('\nModel limitation: because the dependent variable is constructed directly from Henry Hub, the full model is expected to be dominated by gas price. Actual ERCOT hub prices would be needed for a stronger cannibalization test.')


MODEL 1: Simple OLS — Renewable Share -> Electricity Price Proxy
                                  OLS Regression Results                                 
Dep. Variable:     implied_wholesale_price_proxy   R-squared:                       0.015
Model:                                       OLS   Adj. R-squared:                  0.006
Method:                            Least Squares   F-statistic:                     1.717
Date:                           Wed, 27 May 2026   Prob (F-statistic):              0.193
Time:                                   12:08:47   Log-Likelihood:                -552.07
No. Observations:                            120   AIC:                             1108.
Df Residuals:                                118   BIC:                             1114.
Df Model:                                      1                                         
Covariance Type:                             HC1                                         
                    coef    std err

In [4]:
fig = px.scatter(
    price_df.dropna(),
    x='renewable_pct',
    y='implied_wholesale_price_proxy',
    color='season',
    color_discrete_map={'summer': '#E53935', 'winter': '#1E88E5', 'shoulder': '#757575'},
    labels={
        'renewable_pct': 'Renewable Share (% of monthly generation)',
        'implied_wholesale_price_proxy': 'Gas-Implied Price Proxy ($/MWh)',
        'season': 'Season',
    },
    hover_data=['year', 'month', 'henry_hub_price'],
)
fig = apply_chart_standard(
    fig,
    'ERCOT Renewable Share vs Gas-Implied Price Proxy',
    'Renewable Share (% of monthly generation)',
    'Gas-Implied Price Proxy ($/MWh)',
)
fig.add_annotation(
    xref='paper', yref='paper', x=0.02, y=0.98,
    text='Descriptive proxy only:<br>actual ERCOT hub and LMP prices are required<br>to test price cannibalization directly.',
    showarrow=False,
    bgcolor='white',
    bordercolor='black',
    font=dict(size=11),
    align='left',
    xanchor='left',
    yanchor='top',
)
print('Chart 2a is intentionally descriptive. No OLS coefficient is shown because the price proxy is mechanically derived from Henry Hub gas prices.')
save_chart(fig, 'chart_2a_price_cannibalization_scatter')
fig.show()

Chart 2a is intentionally descriptive. No OLS coefficient is shown because the price proxy is mechanically derived from Henry Hub gas prices.
Saved: outputs/charts/chart_2a_price_cannibalization_scatter.html


Saved: outputs/charts/chart_2a_price_cannibalization_scatter.png


In [5]:

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=price_df['month_date'], y=price_df['henry_hub_price'],
    name='Henry Hub Gas Price ($/MMBtu)', line=dict(color='#FF7043', width=2), yaxis='y1'
))
fig.add_trace(go.Scatter(
    x=price_df['month_date'], y=price_df['implied_wholesale_price_proxy'],
    name='Gas-Implied Wholesale Electricity ($/MWh)', line=dict(color='#1E88E5', width=2), yaxis='y2'
))
uri_marker = pd.Timestamp('2021-02-15')
fig.add_shape(type='line', x0=uri_marker, x1=uri_marker, y0=0, y1=1, xref='x', yref='paper', line=dict(color='red', dash='dash'))
fig.add_annotation(x=uri_marker, y=1.06, xref='x', yref='paper', text='Uri spike', showarrow=False, font=dict(size=11, color='red'))
fig.update_layout(
    title=dict(text='ERCOT Gas Price Proxy: Henry Hub vs Implied Electricity Price 2015-2024', font=dict(size=18, family='Arial')),
    xaxis_title='Date',
    yaxis=dict(title='Gas Price ($/MMBtu)', side='left', color='#FF7043', showgrid=True, gridcolor='#F5F5F5'),
    yaxis2=dict(title='Gas-Implied Electricity Price ($/MWh)', side='right', overlaying='y', color='#1E88E5'),
    hovermode='x unified', plot_bgcolor='white', paper_bgcolor='white',
    legend=dict(orientation='h', yanchor='bottom', y=1.02),
    margin=dict(l=70, r=80, t=90, b=80),
)
fig.add_annotation(xref='paper', yref='paper', x=1.0, y=-0.14, text='Source: U.S. EIA; author calculations', showarrow=False, font=dict(size=10, color='gray'), xanchor='right')
save_chart(fig, 'chart_2b_gas_electricity_correlation')
fig.show()


Saved: outputs/charts/chart_2b_gas_electricity_correlation.html


Saved: outputs/charts/chart_2b_gas_electricity_correlation.png
